In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Assignment 1')
import importlib
import utils

importlib.reload(utils)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<module 'utils' from '/content/drive/MyDrive/Colab Notebooks/Assignment 1/utils.py'>

In [8]:
from torch.utils.data import random_split, DataLoader
from utils import LinearModel, MLP
import torch.nn as nn
import torchvision
import torch

In [4]:
transform = torchvision.transforms.ToTensor()
train_set = torchvision.datasets.FashionMNIST(root='./data', transform = transform, download = True, train = True)
test_set = torchvision.datasets.FashionMNIST(root = './data', transform = transform, download = True, train = False)

In [5]:
m_test = len(test_set)
m_val = m_test
m_train = len(train_set) - m_val
total_samples = m_train + m_val + m_test

generator_67 = torch.Generator().manual_seed(67)
xy_train, xy_val = random_split(train_set, [m_train, m_val], generator_67)
print(f"Number of samples in training set: {len(xy_train)}")
print(f"Number of samples in validation set: {len(xy_val)}")
print(f"Number of samples in test: {len(test_set)}")
print(f"Split propotion: {m_train * 100 / total_samples:.2f}/{m_val * 100 / total_samples:.2f}/{m_test * 100 / total_samples:.2f}")

Number of samples in training set: 50000
Number of samples in validation set: 10000
Number of samples in test: 10000
Split propotion: 71.43/14.29/14.29


In [6]:
def fit_compile(model, loss_fn, optimizer, epochs, max_patience, train_loader, val_loader, save_path):
  batch_loss_record = []
  val_accuracy_record = []
  best_val_acc = 0.0
  patience = 0
  for epoch in range(epochs):
    model.train()
    for batchidx, (images, labels) in enumerate(train_loader):
      predictions = model(images)
      loss = loss_fn(predictions, labels)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      batch_loss_record.append(loss.item())
      if batchidx % 100 == 0:
        print(f"Current epoch: {epoch}, current batch: {batchidx}, current loss: {loss.item()}")

    model.eval()
    with torch.no_grad():
      correct = 0
      total = 0
      for images, labels in val_loader:
        predictions = model(images)
        correct += (predictions.argmax(dim = 1) == labels).sum().item()
        total += labels.size(0)
      val_acc = (correct / total) * 100
      print(f"Epoch: {epoch}, Current validation accuracy: {val_acc:.4f}%")
      val_accuracy_record.append(val_acc)
      if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience = 0
        torch.save(model.state_dict(), save_path)
        print(f"New best model is saved at epoch {epoch}")
      else:
        patience += 1
      if patience >= max_patience:
        print(f"Patience exceeded, training is stopped.")
        break
  model.load_state_dict(torch.load(save_path, weights_only = True))
  return model, batch_loss_record, val_accuracy_record


In [12]:
train_loader = DataLoader(dataset = xy_train, batch_size = 64, shuffle = True, generator = generator_67)
val_loader = DataLoader(dataset = xy_val, batch_size = 64, shuffle = False)
lin_model = LinearModel()
epochs = 50
max_patience = 10
lin_path = 'best_linear_model.pth'
SparseCategoricalCrossentropy = nn.CrossEntropyLoss()
adam_optimizer = torch.optim.Adam(lin_model.parameters(), lr = 0.001)

In [8]:
optimized_lin_model, loss_track, validation_track = fit_compile(model = lin_model,
                                                                epochs = epochs,
                                                                max_patience = max_patience,
                                                                optimizer = adam_optimizer,
                                                                loss_fn = SparseCategoricalCrossentropy,
                                                                save_path = lin_path,
                                                                train_loader = train_loader,
                                                                val_loader = val_loader)

Current epoch: 0, current batch: 0, current loss: 2.3432416915893555
Current epoch: 0, current batch: 100, current loss: 0.861179530620575
Current epoch: 0, current batch: 200, current loss: 0.5080618858337402
Current epoch: 0, current batch: 300, current loss: 0.7717680931091309
Current epoch: 0, current batch: 400, current loss: 0.5802299380302429
Current epoch: 0, current batch: 500, current loss: 0.3763522207736969
Current epoch: 0, current batch: 600, current loss: 0.3989204168319702
Current epoch: 0, current batch: 700, current loss: 0.5004721879959106
Epoch: 0, Current validation accuracy: 82.4100%
New best model is saved at epoch 0
Current epoch: 1, current batch: 0, current loss: 0.5502718091011047
Current epoch: 1, current batch: 100, current loss: 0.4721393585205078
Current epoch: 1, current batch: 200, current loss: 0.5258808732032776
Current epoch: 1, current batch: 300, current loss: 0.5210794806480408
Current epoch: 1, current batch: 400, current loss: 0.3609480261802673

In [ ]:
for layer in optimized_lin_model.state_dict():
  print(f"{layer}: {optimized_lin_model.state_dict()[layer].size()}")

output.weight: torch.Size([10, 784])
output.bias: torch.Size([10])


In [ ]:
mlp_model = MLP()
mlp_path = 'best_mlp_model.pth'
mlp_adam_optimizer = torch.optim.Adam(mlp_model.parameters(), lr = 0.001)

In [14]:
best_mlp_model, mlp_loss_track, mlp_val_track = fit_compile(model = mlp_model,
                                                            loss_fn = SparseCategoricalCrossentropy,
                                                            optimizer = mlp_adam_optimizer,
                                                            epochs = epochs,
                                                            max_patience = max_patience,
                                                            train_loader = train_loader,
                                                            val_loader = val_loader,
                                                            save_path = mlp_path)

Current epoch: 0, current batch: 0, current loss: 0.37908029556274414
Current epoch: 0, current batch: 100, current loss: 0.1490238606929779
Current epoch: 0, current batch: 200, current loss: 0.259600967168808
Current epoch: 0, current batch: 300, current loss: 0.18049587309360504
Current epoch: 0, current batch: 400, current loss: 0.3018004596233368
Current epoch: 0, current batch: 500, current loss: 0.13778313994407654
Current epoch: 0, current batch: 600, current loss: 0.2540552318096161
Current epoch: 0, current batch: 700, current loss: 0.17558234930038452
Epoch: 0, Current validation accuracy: 87.8800%
New best model is saved at epoch 0
Current epoch: 1, current batch: 0, current loss: 0.14336630702018738
Current epoch: 1, current batch: 100, current loss: 0.33681410551071167
Current epoch: 1, current batch: 200, current loss: 0.23281461000442505
Current epoch: 1, current batch: 300, current loss: 0.17055200040340424
Current epoch: 1, current batch: 400, current loss: 0.37064895